# 03 — RAG Pipeline

Goal: build a Retrieval-Augmented Generation pipeline that retrieves relevant company policy chunks and generates grounded answers using only the retrieved context.

In [12]:
SYSTEM_PROMPT = """
You are a company knowledge assistant.

Use only the provided context to answer the user's question.
Do not invent, assume, or add company rules, policies, or product details that are not present in the context.

If the answer is not found in the provided context, respond with:
"No such information is available in the provided context."

Answer in a concise and professional way.

When possible, include the relevant document title, document ID, or chunk ID as a source.
"""


### System Prompt Requirements

The system prompt defines the assistant's behavior.  
For this RAG assistant, the most important requirements are grounding, no hallucination, concise answers, and source attribution.

In [13]:
USER_PROMPT_TEMPLATE = """
Use the following retrieved context to answer the question.

Context:
{context}

Question:
{question}

Instructions:
- Answer using only the context above.
- If the answer is not found in the context, say: "No such information is available in the provided context."
- Keep the answer concise and professional.
- Include the most relevant source title, document ID, or chunk ID when available.

answer:
"""


def build_user_prompt(question: str, retrieved_chunks: list[dict]) -> str:
    """Create the user prompt by combining retrieved context with the user's question."""
    context_blocks = []

    for chunk in retrieved_chunks:
        context_blocks.append(
            f"[{chunk['chunk_id']}]\n"
            f"Title: {chunk['title']}\n"
            f"Document ID: {chunk['doc_id']}\n"
            f"Source: {chunk['source']}\n"
            f"Text: {chunk['text']}"
        )

    context = "\n\n---\n\n".join(context_blocks)

    return USER_PROMPT_TEMPLATE.format(
        context=context,
        question=question,
    )

### User Prompt Requirements

The user prompt carries the retrieved chunks and the actual user question. It should clearly separate context from the question so the model knows what information it is allowed to use.

In [14]:
example_chunks = [
    {
        "chunk_id": "HR-001-CHUNK-001",
        "doc_id": "HR-001",
        "title": "Vacation and Paid Time Off Policy",
        "source": "HR Handbook v2.1",
        "text": "Employees receive 21 paid vacation days per calendar year.",
    }
]

example_question = "How many vacation days do employees get?"

print(build_user_prompt(example_question, example_chunks))


Use the following retrieved context to answer the question.

Context:
[HR-001-CHUNK-001]
Title: Vacation and Paid Time Off Policy
Document ID: HR-001
Source: HR Handbook v2.1
Text: Employees receive 21 paid vacation days per calendar year.

Question:
How many vacation days do employees get?

Instructions:
- Answer using only the context above.
- If the answer is not found in the context, say: "No such information is available in the provided context."
- Keep the answer concise and professional.
- Include the most relevant source title, document ID, or chunk ID when available.

answer:



### Assistant Style Rules

These rules define how the assistant should shape the final answer after retrieval.

In [15]:
ASSISTANT_STYLE_RULES = """
Citations:
- Cite the source for every factual answer using the document title and document ID.
- Include the chunk ID when it helps trace the exact retrieved passage.
- Do not cite sources that were not included in the retrieved context.
- If multiple retrieved sources support the answer, cite the most relevant one first.

Answer length:
- Keep answers short: usually 1-3 sentences.
- Use bullets only when the answer contains multiple distinct items, rules, or steps.
- Do not include long explanations unless the user asks for more detail.

Formatting:
- Start with the direct answer.
- Put source attribution at the end of the answer.
- Use plain language and avoid unnecessary technical terms.

Tone:
- Be professional, calm, and helpful.
- Do not sound overly casual or conversational.
- Do not speculate about company policy.

Handling missing info:
- If the retrieved context does not contain the answer, say exactly: "No such information is available in the provided context."
- Do not use outside knowledge to fill gaps.
- Do not guess, infer, or create rules that are not explicitly supported by the context.
"""


FULL_SYSTEM_PROMPT = f"{SYSTEM_PROMPT}\n\n{ASSISTANT_STYLE_RULES}"

print(FULL_SYSTEM_PROMPT)


You are a company knowledge assistant.

Use only the provided context to answer the user's question.
Do not invent, assume, or add company rules, policies, or product details that are not present in the context.

If the answer is not found in the provided context, respond with:
"No such information is available in the provided context."

Answer in a concise and professional way.

When possible, include the relevant document title, document ID, or chunk ID as a source.



Citations:
- Cite the source for every factual answer using the document title and document ID.
- Include the chunk ID when it helps trace the exact retrieved passage.
- Do not cite sources that were not included in the retrieved context.
- If multiple retrieved sources support the answer, cite the most relevant one first.

Answer length:
- Keep answers short: usually 1-3 sentences.
- Use bullets only when the answer contains multiple distinct items, rules, or steps.
- Do not include long explanations unless the user 

### Test Zero-Shot Prompting

Zero-shot prompting means we give the model instructions, context, and a question, but no example answer.

In [16]:
zero_shot_question = "How many vacation days do employees get?"

zero_shot_chunks = [
    {
        "chunk_id": "HR-001-CHUNK-001",
        "doc_id": "HR-001",
        "title": "Vacation and Paid Time Off Policy",
        "source": "HR Handbook v2.1",
        "text": "Employees receive 21 paid vacation days per calendar year. Vacation days accrue monthly starting from the employee's hire date.",
    }
]

zero_shot_user_prompt = build_user_prompt(zero_shot_question, zero_shot_chunks)

print("SYSTEM PROMPT")
print(FULL_SYSTEM_PROMPT)
print("=" * 80)
print("USER PROMPT")
print(zero_shot_user_prompt)

SYSTEM PROMPT

You are a company knowledge assistant.

Use only the provided context to answer the user's question.
Do not invent, assume, or add company rules, policies, or product details that are not present in the context.

If the answer is not found in the provided context, respond with:
"No such information is available in the provided context."

Answer in a concise and professional way.

When possible, include the relevant document title, document ID, or chunk ID as a source.



Citations:
- Cite the source for every factual answer using the document title and document ID.
- Include the chunk ID when it helps trace the exact retrieved passage.
- Do not cite sources that were not included in the retrieved context.
- If multiple retrieved sources support the answer, cite the most relevant one first.

Answer length:
- Keep answers short: usually 1-3 sentences.
- Use bullets only when the answer contains multiple distinct items, rules, or steps.
- Do not include long explanations un

In [17]:
zero_shot_expected_answer = "Employees receive 21 paid vacation days per calendar year. Source: Vacation and Paid Time Off Policy (HR-001, HR-001-CHUNK-001)."

print(zero_shot_expected_answer)

Employees receive 21 paid vacation days per calendar year. Source: Vacation and Paid Time Off Policy (HR-001, HR-001-CHUNK-001).


### Test One-Shot Prompting

One-shot prompting means we include one example question and answer before asking the real question. This helps the model copy the answer style, citation format, and fallback behavior.

In [18]:
ONE_SHOT_EXAMPLE = """
Example:

Context:
[IT-001-CHUNK-001]
Title: Password and Multi-Factor Authentication Policy
Document ID: IT-001
Source: IT Security Standards v3.0
Text: Passwords must contain at least 12 characters including uppercase letters, lowercase letters, numbers, and symbols.

Question:
What are the password length requirements?

answer:
Passwords must contain at least 12 characters. Source: Password and Multi-Factor Authentication Policy (IT-001, IT-001-CHUNK-001).
"""


ONE_SHOT_SYSTEM_PROMPT = f"{FULL_SYSTEM_PROMPT}\n\n{ONE_SHOT_EXAMPLE}"

one_shot_question = "When do VPN sessions disconnect?"

one_shot_chunks = [
    {
        "chunk_id": "IT-003-CHUNK-001",
        "doc_id": "IT-003",
        "title": "VPN and Secure Access Guidelines",
        "source": "Network Security Handbook",
        "text": "VPN sessions automatically disconnect after 30 minutes of inactivity.",
    }
]

one_shot_user_prompt = build_user_prompt(one_shot_question, one_shot_chunks)

print("SYSTEM PROMPT WITH ONE EXAMPLE")
print(ONE_SHOT_SYSTEM_PROMPT)
print("=" * 80)
print("USER PROMPT")
print(one_shot_user_prompt)

SYSTEM PROMPT WITH ONE EXAMPLE

You are a company knowledge assistant.

Use only the provided context to answer the user's question.
Do not invent, assume, or add company rules, policies, or product details that are not present in the context.

If the answer is not found in the provided context, respond with:
"No such information is available in the provided context."

Answer in a concise and professional way.

When possible, include the relevant document title, document ID, or chunk ID as a source.



Citations:
- Cite the source for every factual answer using the document title and document ID.
- Include the chunk ID when it helps trace the exact retrieved passage.
- Do not cite sources that were not included in the retrieved context.
- If multiple retrieved sources support the answer, cite the most relevant one first.

Answer length:
- Keep answers short: usually 1-3 sentences.
- Use bullets only when the answer contains multiple distinct items, rules, or steps.
- Do not include lon

In [19]:
one_shot_expected_answer = "VPN sessions automatically disconnect after 30 minutes of inactivity. Source: VPN and Secure Access Guidelines (IT-003, IT-003-CHUNK-001)."

print(one_shot_expected_answer)

VPN sessions automatically disconnect after 30 minutes of inactivity. Source: VPN and Secure Access Guidelines (IT-003, IT-003-CHUNK-001).


### Prompting Test Notes

- The zero-shot prompt relies only on instructions and retrieved context.
- The one-shot prompt adds one example, which can make citation format and answer length more consistent.
- In both cases, the answer must come only from retrieved context. If the context does not contain the answer, the assistant should use the missing-information fallback.